In [ ]:
import os, pathlib, time
from datetime import timedelta

finished_files = pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline").glob(
    "cad/*/*/done-reflbake.success"
)
data = []
for ff in finished_files:
    fd = ff.parent
    bakery = fd / "bakery"
    baked_files = list(bakery.glob("*RawReflection*"))
    mtimes = [os.path.getmtime(bf) for bf in baked_files]

    # Get the duration of each object
    objs_in_bake_order = sorted(zip(baked_files, mtimes), key=lambda x: x[1])
    for i, (bf, mtime) in enumerate(objs_in_bake_order):
        if i == 0:
            continue
        prev_bf, prev_mtime = objs_in_bake_order[i - 1]
        duration = mtime - prev_mtime
        assert (
            duration < 60 * 60
        ), f"Duration longer than 1h for {bf} (prev: {prev_bf}, duration: {duration})"
        data.append(
            {
                "file": str(fd.name),
                "object": str(bf.name),
                "duration": timedelta(seconds=duration),
            }
        )

import pandas as pd

df = pd.DataFrame(data)

In [ ]:
# Print file-level stats
df.groupby("file").agg(
    {"duration": ["count", "mean", "min", "max", "sum"]}
).sort_values(by=("duration", "mean"), ascending=False)

In [ ]:
# Print general statistics of the durations
print(f"Mean duration: {df['duration'].mean()}")
print(f"Min duration: {df['duration'].min()}")
print(f"Max duration: {df['duration'].max()}")

In [ ]:
# Plot a histogram of the durations
import matplotlib.pyplot as plt

plt.hist(df["duration"], bins=50, edgecolor="black")
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Distribution of Object Bake Durations")
plt.show()

In [ ]:
# What are the 10 worst offenders (longest bake times)?
worst_offenders = df.sort_values(by="duration", ascending=False).head(10)
print("Top 10 longest bake times:")
worst_offenders